# Analyse des Greeks - Options Call et Put

Ce notebook calcule et visualise les Greeks (sensibilités) des options européennes vanille.

**Greeks calculés :**
- Delta (∂V/∂S) : sensibilité au prix du sous-jacent
- Gamma (∂²V/∂S²) : variation du Delta
- Vega (∂V/∂σ) : sensibilité à la volatilité
- Theta (∂V/∂t) : érosion temporelle
- Rho (∂V/∂r) : sensibilité au taux d'intérêt
- Vanna (∂²V/∂S∂σ) : variation du Delta quand la volatilité change

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
import math
%matplotlib inline

In [ ]:
# On choisit des valeurs pour un call. On fait varier le spot entre 50 et 170 et on choisit plusieurs maturités à observer

S = np.linspace(50, 170, 200)
K = 100
sigma = 0.2
maturities = [1.0, 0.5, 0.25, 0.1, 0.02]
r = 0.05

In [ ]:
# Fonction prix Black-Scholes Call
def call_price(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

In [ ]:
# Fonction prix Black-Scholes Put
def put_price(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

In [ ]:
# Pas pour dérivation numérique
h = 0.01

---
## 1. DELTA (∂V/∂S)

### Définition
Le **Delta** mesure la variation du prix de l'option pour un mouvement de 1% du spot.

**Formule :** ΔCall = ∂C/∂S  |  ΔPut = ∂P/∂S

### Allure de la courbe
**Call :** Delta ∈ [0, 1]
- Au début la PV est plate (OTM) donc Delta = 0
- Ensuite Delta > 0 car PV devient croissante
- Delta → 1 car PV devient affine (S - K) quand très ITM

**Put :** Delta ∈ [-1, 0]
- Symétrique au call mais négatif
- Delta → -1 quand très ITM (S << K)

### Delta = 0.5 quand K = Spot Forward
Le Delta peut être **approximé comme la probabilité d'exercer l'option**.

Donc : Delta(K = S_ATM_forward) = 0.5

**Spot forward :** F = S × e^(rT)

Quand K = Forward, on doit avoir Delta = 0.5. On le vérifie ci-dessous.

### À quoi sert le Delta ?
- **Delta-hedging** : couvrir le risque directionnel en achetant/vendant Delta actions
- **Probabilité d'exercice** : |Delta| ≈ probabilité que l'option finisse ITM
- **Équivalent actions** : 1 call Delta=0.6 ≈ détenir 60 actions

In [ ]:
# Delta : dPrix/dSpot

# Vérification : quand K = Forward, Delta doit être = 0.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# CALL
for T in maturities:
    # Calcul du spot forward pour cette maturité
    fwd = K * np.exp(-r * T)
    print(f"Spot fwd pour T = {T}")
    print(fwd)
    
    delta_call = (call_price(S + h, K, r, sigma, T) - 
                  call_price(S - h, K, r, sigma, T)) / (2 * h)
    ax1.plot(S, delta_call, label=f"T = {T}")

ax1.axhline(y=0.5, color='r', linestyle='--', linewidth=0.8, alpha=0.5, label='Delta = 0.5')
ax1.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label=f'K = {K}')
ax1.set_xlabel("Spot")
ax1.set_ylabel("Delta du Call")
ax1.set_title("Convergence du Delta - CALL")
ax1.legend()
ax1.grid()

# PUT
for T in maturities:
    delta_put = (put_price(S + h, K, r, sigma, T) - 
                 put_price(S - h, K, r, sigma, T)) / (2 * h)
    ax2.plot(S, delta_put, label=f"T = {T}")

ax2.axhline(y=-0.5, color='r', linestyle='--', linewidth=0.8, alpha=0.5, label='Delta = -0.5')
ax2.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label=f'K = {K}')
ax2.set_xlabel("Spot")
ax2.set_ylabel("Delta du Put")
ax2.set_title("Convergence du Delta - PUT")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

In [ ]:
# Tableau : S_ATM_forward (K = Forward) et Delta Call/Put pour chaque maturité
# Condition : K = F = S * e^(rT)  =>  S_atm_fwd = K * e^(-rT)

rows = []
for T in maturities:
    S_atm_fwd = K * math.exp(-r * T)  # spot tel que forward = K
    
    delta_call_atm = (call_price(S_atm_fwd + h, K, r, sigma, T) - 
                      call_price(S_atm_fwd - h, K, r, sigma, T)) / (2 * h)
    
    delta_put_atm  = (put_price(S_atm_fwd + h, K, r, sigma, T) - 
                      put_price(S_atm_fwd - h, K, r, sigma, T)) / (2 * h)
    
    rows.append({
        "Maturité T": T,
        "S_ATM_forward  (K=Fwd)": round(S_atm_fwd, 4),
        "Delta Call": round(delta_call_atm, 4),
        "Delta Put": round(delta_put_atm, 4),
        "Delta Call ≈ 0.5 ?": "✓" if abs(delta_call_atm - 0.5) < 0.05 else "✗",
        "Delta Put ≈ -0.5 ?": "✓" if abs(delta_put_atm + 0.5) < 0.05 else "✗",
    })

df_atm = pd.DataFrame(rows).set_index("Maturité T")
print("Vérification : quand S = K·e^(-rT), le forward vaut K et Delta ≈ ±0.5\n")
df_atm

---
## 2. GAMMA (∂²V/∂S²)

### Définition
Le **Gamma** mesure la variation du Delta pour un mouvement de 1€ du sous-jacent.

**Formule :** Γ = ∂²V/∂S² = ∂Δ/∂S

### Allure de la courbe
- **Symétrique** autour de K (Gamma Call = Gamma Put)
- **Maximum à ATM** (S = K) : le Delta change le plus rapidement
- **Proche de 0 sur les bords** (OTM/ITM profond) : le Delta ne bouge plus
- **Pic très aigu** quand T → 0 : le Delta passe brutalement de 0 à 1 à l'échéance

### Pourquoi cette forme ?
Le Gamma est la dérivée du Delta, qui lui-même est une courbe en S.
La dérivée d'une sigmoïde donne une courbe en cloche (fonction de densité normale).
À maturité courte, la transition est brutale → pic très étroit et haut.

### À quoi sert le Gamma ?
- **Erreur de hedge** : si Gamma élevé, le Delta change vite → il faut rebalancer souvent
- **Scalping** : traders profitent du Gamma en rebalançant quand le marché bouge
- **P&L non-linéaire** : ΔP&L ≈ Delta×ΔS + 0.5×Gamma×(ΔS)²
- **Risque à l'échéance** : Gamma explosif proche de T=0 pour options ATM

In [ ]:
# Gamma = d²Prix / dSpot²

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# CALL
for T in maturities:
    gamma_call = (call_price(S + h, K, r, sigma, T) - 
                  2 * call_price(S, K, r, sigma, T) + 
                  call_price(S - h, K, r, sigma, T)) / (h**2)
    ax1.plot(S, gamma_call, label=f"T = {T}")

ax1.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.set_xlabel("Spot")
ax1.set_ylabel("Gamma du Call")
ax1.set_title("Convergence du Gamma - CALL")
ax1.legend()
ax1.grid()

# PUT
for T in maturities:
    gamma_put = (put_price(S + h, K, r, sigma, T) - 
                 2 * put_price(S, K, r, sigma, T) + 
                 put_price(S - h, K, r, sigma, T)) / (h**2)
    ax2.plot(S, gamma_put, label=f"T = {T}")

ax2.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.set_xlabel("Spot")
ax2.set_ylabel("Gamma du Put")
ax2.set_title("Convergence du Gamma - PUT")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

---
## 3. VEGA (∂V/∂σ)

### Définition
Le **Vega** mesure la variation du prix de l'option pour un mouvement de 1% de volatilité.

**Formule :** ν = ∂V/∂σ

### Allure de la courbe
- **Vega Call = Vega Put** (sensibilité identique)
- **Maximum à ATM** (S = K) : l'option est la plus sensible à la volatilité
- **Proche de 0 sur les bords** (OTM/ITM profond) : les mouvements de vol n'ont pas d'impact
- **Plus élevé pour maturités longues** : plus de temps = plus d'incertitude = plus sensible à σ

### Pourquoi cette forme ?
Une option ATM a 50% de chances de finir ITM → l'incertitude (volatilité) impacte beaucoup sa valeur.
Une option très ITM finira ITM quoi qu'il arrive → la volatilité ne change rien.
Plus l'échéance est lointaine, plus la volatilité a le temps de "s'exprimer" → Vega augmente avec T.

### À quoi sert le Vega ?
- **Trading de volatilité** : acheter/vendre options en anticipant les mouvements de vol implicite
- **Vega-neutral portfolios** : se couvrir contre les variations de volatilité
- **Earnings/événements** : avant résultats trimestriels, la vol implicite monte → Vega positif gagne
- **Vol smile** : gérer l'exposition à différents strikes selon la skew de volatilité

In [ ]:
# Vega = dPrix / dVolatilité

# Vega Call = Vega Put
# Maximale quand le call est ATM, car il sera le plus sensible à la vol
# = 0 sur les bords car si le call est OTM ou ITM profond, les mouvements de vol n'auront pas d'impact
# Plus la maturité est longue, plus le Vega est élevé. Pour les options de faible maturité, la vol n'a pas tant que ça d'importance

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# CALL
for T in maturities:
    vega_call = (call_price(S, K, r, sigma + h, T) - 
                 call_price(S, K, r, sigma - h, T)) / (2 * h)
    ax1.plot(S, vega_call, label=f"T = {T}")

ax1.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.set_xlabel("Spot")
ax1.set_ylabel("Vega du Call")
ax1.set_title("Vega en fonction du Spot - CALL")
ax1.legend()
ax1.grid()

# PUT
for T in maturities:
    vega_put = (put_price(S, K, r, sigma + h, T) - 
                put_price(S, K, r, sigma - h, T)) / (2 * h)
    ax2.plot(S, vega_put, label=f"T = {T}")

ax2.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.set_xlabel("Spot")
ax2.set_ylabel("Vega du Put")
ax2.set_title("Vega en fonction du Spot - PUT")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

---
## 4. THETA (∂V/∂t)

### Définition
Le **Theta** mesure l'érosion temporelle du prix de l'option : combien l'option perd de valeur chaque jour qui passe.

**Formule :** Θ = -∂V/∂t  (on affiche -Theta car Theta < 0)

### Allure de la courbe
- **Theta < 0** pour options achetées (long) : le temps joue contre vous
- **Maximum à ATM** : une option ATM perd le plus de valeur temps chaque jour
- **Faible sur les bords** (OTM/ITM profond) : ces options ont peu de valeur temps à perdre
- **Accélération proche de l'échéance** : T → 0, l'érosion s'accélère (courbe T=0.02 très élevée)

### Pourquoi cette forme ?
Une option = valeur intrinsèque + valeur temps.
À ATM, toute la valeur est temporelle → Theta maximum.
Plus l'échéance approche, plus il reste peu de temps à perdre → le rythme de perte s'accélère.

### À quoi sert le Theta ?
- **Option selling strategies** : vendre des options pour capturer le Theta positif (ex: covered calls)
- **Time decay management** : savoir combien on perd chaque jour sur une position longue
- **Calendar spreads** : jouer la différence de Theta entre maturités courtes et longues
- **Éviter les weekends** : 3 jours de Theta perdus d'un coup le vendredi soir

In [ ]:
# Theta = -dPrix / dTemps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# CALL
for T in maturities:
    theta_call = (call_price(S, K, r, sigma, T + h) - 
                  call_price(S, K, r, sigma, T - h)) / (2 * h)
    ax1.plot(S, -theta_call, label=f"T = {T}")  # -Theta car Theta < 0

ax1.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.set_xlabel("Spot")
ax1.set_ylabel("Theta du Call (perte par jour)")
ax1.set_title("Convergence du Theta - CALL")
ax1.legend()
ax1.grid()

# PUT
for T in maturities:
    theta_put = (put_price(S, K, r, sigma, T + h) - 
                 put_price(S, K, r, sigma, T - h)) / (2 * h)
    ax2.plot(S, -theta_put, label=f"T = {T}")

ax2.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.set_xlabel("Spot")
ax2.set_ylabel("Theta du Put (perte par jour)")
ax2.set_title("Convergence du Theta - PUT")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

---
## 5. RHO (∂V/∂r)

### Définition
Le **Rho** mesure la variation du prix de l'option pour un mouvement de 1% du taux d'intérêt sans risque.

**Formule :** ρ = ∂V/∂r

### Allure de la courbe
- **Rho Call > 0** : hausse des taux → prix call augmente
- **Rho Put < 0** : hausse des taux → prix put diminue
- **Croissant avec S** pour call, décroissant pour put
- **Plus élevé pour maturités longues** : plus de temps = plus d'actualisation = plus sensible à r

### Pourquoi cette forme ?
Le prix d'une option contient K×e^(-rT) (actualisation du strike).
Pour un call : si r↑ → K actualisé↓ → call vaut plus cher (payer K dans le futur coûte moins cher aujourd'hui).
Pour un put : si r↑ → K actualisé↓ → put vaut moins cher (recevoir K dans le futur vaut moins aujourd'hui).

### À quoi sert le Rho ?
- **Moins important que les autres Greeks** : les taux bougent lentement vs prix/volatilité
- **Couverture macro** : se protéger contre les décisions des banques centrales (Fed, BCE)
- **Options long-terme (LEAPS)** : Rho devient significatif pour T > 2 ans
- **Arbitrage de taux** : exploiter les différences de taux entre marchés

In [ ]:
# Rho = dPrix / dTaux

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# CALL
for T in maturities:
    rho_call = (call_price(S, K, r + h, sigma, T) - 
                call_price(S, K, r - h, sigma, T)) / (2 * h)
    ax1.plot(S, rho_call, label=f"T = {T}")

ax1.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xlabel("Spot")
ax1.set_ylabel("Rho du Call")
ax1.set_title("Rho en fonction du Spot - CALL")
ax1.legend()
ax1.grid()

# PUT
for T in maturities:
    rho_put = (put_price(S, K, r + h, sigma, T) - 
               put_price(S, K, r - h, sigma, T)) / (2 * h)
    ax2.plot(S, rho_put, label=f"T = {T}")

ax2.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel("Spot")
ax2.set_ylabel("Rho du Put")
ax2.set_title("Rho en fonction du Spot - PUT")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

---
## 6. VANNA (∂²V/∂S∂σ)

### Définition
Le **Vanna** mesure la variation du Delta quand la volatilité change (ou équivalent : variation du Vega quand le spot change).

**Formule :** Vanna = ∂²V/∂S∂σ = ∂Δ/∂σ = ∂ν/∂S

### Allure de la courbe
- **Vanna Call = Vanna Put** (sensibilité identique)
- **Positif pour S < K, négatif pour S > K** (change de signe autour de ATM)
- **Maximum en valeur absolue légèrement OTM/ITM** (pas exactement à ATM)
- **Nul à ATM et sur les bords** (OTM/ITM profond)

### Pourquoi cette forme ?
Le Vanna capture l'interaction entre spot et volatilité.
Quand S bouge, le moneyness change → la sensibilité à la volatilité (Vega) change aussi.
Vanna change de signe car :
- Si option devient plus ITM → moins sensible à la vol → Vega baisse
- Si option devient plus OTM → moins sensible à la vol → Vega baisse

### À quoi sert le Vanna ?
- **Greek de second ordre** (moins utilisé que Delta/Gamma/Vega)
- **Vol smile hedging** : gérer l'exposition croisée spot-volatilité
- **Exotic options** : crucial pour les barrières, autocalls, worst-of
- **Market making** : ajuster dynamiquement le hedge quand vol et spot bougent ensemble
- **Stress scenarios** : modéliser les pertes quand spot baisse ET volatilité monte (crise)

In [ ]:
# Vanna = d²Prix / (dSpot × dVolatilité)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# CALL
for T in maturities:
    vanna_call = (call_price(S + h, K, r, sigma + h, T) - 
                  call_price(S + h, K, r, sigma - h, T) -
                  call_price(S - h, K, r, sigma + h, T) + 
                  call_price(S - h, K, r, sigma - h, T)) / (4 * h**2)
    ax1.plot(S, vanna_call, label=f"T = {T}")

ax1.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.set_xlabel("Spot")
ax1.set_ylabel("Vanna du Call")
ax1.set_title("Vanna en fonction du Spot - CALL")
ax1.legend()
ax1.grid()

# PUT
for T in maturities:
    vanna_put = (put_price(S + h, K, r, sigma + h, T) - 
                 put_price(S + h, K, r, sigma - h, T) -
                 put_price(S - h, K, r, sigma + h, T) + 
                 put_price(S - h, K, r, sigma - h, T)) / (4 * h**2)
    ax2.plot(S, vanna_put, label=f"T = {T}")

ax2.axvline(x=K, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel("Spot")
ax2.set_ylabel("Vanna du Put")
ax2.set_title("Vanna en fonction du Spot - PUT")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

---
## Résumé des Greeks

| Greek | Formule | Call | Put | Usage principal |
|-------|---------|------|-----|----------------|
| **Delta** | ∂V/∂S | [0, 1] | [-1, 0] | Delta-hedging, probabilité ITM |
| **Gamma** | ∂²V/∂S² | > 0 | > 0 | Erreur de hedge, scalping |
| **Vega** | ∂V/∂σ | > 0 | > 0 | Trading de volatilité |
| **Theta** | -∂V/∂t | < 0 | < 0 | Érosion temporelle, time decay |
| **Rho** | ∂V/∂r | > 0 | < 0 | Sensibilité aux taux (long-term) |
| **Vanna** | ∂²V/∂S∂σ | ± | ± | Vol smile hedging, exotiques |

### Points clés
- **Gamma = Vega = identiques** pour call et put
- **Delta, Rho = opposés** entre call et put
- **Greeks de 1er ordre** (Delta, Vega, Theta, Rho) : sensibilités directes
- **Greeks de 2e ordre** (Gamma, Vanna) : variations des sensibilités
- **Plus ATM + proche échéance = Greeks les plus élevés** (sauf Rho)